In [1]:
# Connecting to GitHub

setwd("~/stat_app")

In [2]:
# Installing and importing libraries

install.packages("stargazer")
install.packages("fixest")

library(haven)
library(sandwich)
library(stargazer)
library(fixest)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


Please cite as: 


 Hlavac, Marek (2022). stargazer: Well-Formatted Regression and Summary Statistics Tables.

 R package version 5.2.3. https://CRAN.R-project.org/package=stargazer 




In [3]:
# Importing data

wave2 <- read_dta("waveII.dta") # midline
wave3 <- read_dta("waveIII.dta") # endline (cf. pag. 13)

Let's remember our econometric equation:

$Y_{icu} = \alpha + \beta_1 I_c + \beta_2 E_c + \beta_3 (I_c \times E_c) + \beta_4' X_{ic} + \epsilon_{icu}$

where $Y_{icu}$ is the outcome for person $i$ in community $c$ and union $u$, $I_c$ is assignment of community $c$ to the incentive program, $E_c$ is assignment of community $c$ to the empowerment program, and $X_{ic}$ is a vector of individual and community controls measured at baseline for strata, age indicators, household size, the presence of an older unmarried sister in the household, school enrollment, mother’s level of education, and whether the community is accessible via public transport (cf. pag. 16).

In [4]:
# Making the baseline vector, X_ic

controls <- c("older_sister", "bl_still_in_school", "bl_education_mother", "bl_HHsize", "bl_public_transit", "bl_age10", "bl_age11", "bl_age12", "bl_age13",
              "bl_age14", "bl_age15", "bl_age16", "bl_age17", "older_sister_miss", "bl_still_in_school_miss", "bl_education_mother_miss", "bl_HHsize_miss", 
              "bl_public_transit_miss")

In [5]:
# Preparating the regression

eq <- paste("anyemp + anyoil + oil_kk + factor(third) + factor(unionID) + ", controls, collapse = " + ")

#eq <- paste("anyemp + anyoil + oil_kk + factor(third) + ", controls, collapse = " + ")

In [6]:
# Sampling at the midline.

dfwave2 <- wave2[wave2$midline == 1 & wave2$washedout == 0 & wave2$before_miss == 0 & wave2$bl_age_reported >= 14 & wave2$bl_age_reported <= 16,]

dfwave2_15 <- dfwave2[dfwave2$bl_age_reported == 14,]

# Running some regressions

f_mmid <- as.formula(paste0("ml_ever_married ~ ", eq))

regr4 <- lm(f_mmid, data = dfwave2)
regr5 <- lm(f_mmid, data = dfwave2_15)

#f_mmid <- as.formula(paste0("ml_ever_married ~ ", eq, " | unionID"))

#regr4 <- feols(f_mmid, data = dfwave2, cluster = ~CLUSTER)
#regr5 <- feols(f_mmid, data = dfwave2_15,  cluster = ~CLUSTER)

In [7]:
# Sampling at the endline

dfwave3 <- wave3[wave3$endline == 1 & wave3$washedout == 0 & wave3$before_miss == 0 & wave3$bl_ever_married == 0 & wave3$bl_age_reported >= 14
                  & wave3$bl_age_reported <= 16,]

dfwave3_15 <- dfwave3[dfwave3$bl_age_reported == 14,]

# Running some regressions

f_18 <- as.formula(paste0("under_18 ~ ", eq))
f_16 <- as.formula(paste0("under_16 ~ ", eq))
f_mend <- as.formula(paste0("ever_married ~ ", eq))
f_mage <- as.formula(paste0("marriage_age ~ ", eq))
f_20 <- as.formula(paste0("ever_birth_20 ~ ", eq))

regr1 <- lm(f_18,  data = dfwave3)
regr2 <- lm(f_18,  data = dfwave3_15)
regr3 <- lm(f_16,  data = dfwave3_15)
regr6 <- lm(f_mend,  data = dfwave3)
regr7 <- lm(f_mend,  data = dfwave3_15)
regr8 <- lm(f_mage, data = dfwave3)
regr9 <- lm(f_mage, data = dfwave3_15)
regr10 <- lm(f_20, data = dfwave3)
regr11 <- lm(f_20, data = dfwave3_15)

#f_18 <- as.formula(paste0("under_18 ~ ", eq, " | unionID"))
#f_16 <- as.formula(paste0("under_16 ~ ", eq, " | unionID"))
#f_mend <- as.formula(paste0("ever_married ~ ", eq, " | unionID"))
#f_mage <- as.formula(paste0("marriage_age ~ ", eq, " | unionID"))
#f_20 <- as.formula(paste0("ever_birth_20 ~ ", eq, " | unionID"))

#regr1 <- feols(f_18,  data = dfwave3, cluster = ~CLUSTER)
#regr2 <- feols(f_18,  data = dfwave3_15,  cluster = ~CLUSTER)
#regr3 <- feols(f_16,  data = dfwave3_15,  cluster = ~CLUSTER)
#regr6 <- feols(f_mend,  data = dfwave3, cluster = ~CLUSTER)
#regr7 <- feols(f_mend,  data = dfwave3_15, cluster = ~CLUSTER)
#regr8 <- feols(f_mage, data = dfwave3, cluster = ~CLUSTER)
#regr9 <- feols(f_mage, data = dfwave3_15,  cluster = ~CLUSTER)
#regr10 <- feols(f_20, data = dfwave3, cluster = ~CLUSTER)
#regr11 <- feols(f_20, data = dfwave3_15,  cluster = ~CLUSTER)

In [8]:
# Table

models <- mget(paste0("regr", 1:11))

se_list <- lapply(models, function(m) {sqrt(diag(vcovCL(m, cluster = ~CLUSTER, type = "HC1")))})

stargazer(models, se = se_list, type = "text", keep = c("anyemp", "anyoil", "oil_kk"), omit = c("factor\\(third\\)", "factor\\(unionID\\)"), 
          omit.stat = c("f", "ser", "rsq", "adj.rsq"))

#etable(regr1, regr2, regr3, regr4, regr5, regr6, regr7, regr8, regr9, regr10, regr11, keep = c("%anyemp","%anyoil","%oil_kk"), 
       #dict = c(anyemp = "Empowerment", anyoil = "Incentive", oil_kk = "Incen. * Empow."))


                                                    Dependent variable:                                       
             -------------------------------------------------------------------------------------------------
                  under_18       under_16  ml_ever_married   ever_married     marriage_age     ever_birth_20  
                (1)       (2)      (3)      (4)      (5)      (6)     (7)     (8)      (9)     (10)     (11)  
--------------------------------------------------------------------------------------------------------------
anyemp        -0.007    -0.005    0.006    0.011    0.009    0.006   0.002   0.012    0.003    0.006   0.005  
              (0.008)   (0.015)  (0.009)  (0.011)  (0.017)  (0.008) (0.012) (0.040)  (0.065)  (0.007) (0.013) 
                                                                                                              
anyoil       -0.049*** -0.074*** -0.020*  -0.025* -0.054*** -0.009  -0.018  0.210*** 0.323*** -0.016* -0.039**
